# Getty JSON to Artworks CSV

Convert Getty Linked Art JSON into the same artworks CSV schema used by the other museum processors.

Output columns:
- `id`
- `artist`
- `title`
- `alt_text`
- `description`
- `date_created`
- `date_acquired_or_updated`
- `institution`
- `medium`
- `image_url`

In [21]:
import json
import re
import html
import ast
import pandas as pd
import os

In [22]:
def clean_text(value):
    if value is None or pd.isna(value):
        return ""
    return html.unescape(str(value)).replace("\xa0", " ").strip()


def smart_join(parts):
    cleaned = [clean_text(p).rstrip(".") for p in parts if clean_text(p)]
    return ". ".join(cleaned)


def as_list(value):
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


def unique_preserve(values):
    out = []
    seen = set()
    for value in values:
        key = clean_text(value)
        if not key:
            continue
        dedupe_key = re.sub(r"\s+", " ", key).lower()
        if dedupe_key in seen:
            continue
        seen.add(dedupe_key)
        out.append(key)
    return out


def extract_year(value):
    text = clean_text(value)
    match = re.search(r"(?<!\d)(1[89]\d{2}|20\d{2}|21\d{2})(?!\d)", text)
    return match.group(1) if match else ""


def load_loose_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        raw = f.read().strip()

    if not raw:
        return {}

    # Strict JSON first.
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    text = raw

    # Handle common JS wrappers.
    text = re.sub(r"^\s*export\s+default\s+", "", text)
    text = re.sub(r"^\s*module\.exports\s*=\s*", "", text)
    text = re.sub(r"^\s*(const|let|var)\s+[A-Za-z_$][\w$]*\s*=\s*", "", text)
    text = text.strip().rstrip(";").strip()

    # JSON Lines fallback (one JSON object per line).
    lines = [ln.strip().rstrip(",") for ln in text.splitlines() if ln.strip()]
    if len(lines) > 1 and all(ln.startswith("{") and ln.endswith("}") for ln in lines):
        try:
            return [json.loads(ln) for ln in lines]
        except json.JSONDecodeError:
            pass

    # Python-literal style fallback for JS-like booleans/null.
    py_text = re.sub(r"\btrue\b", "True", text)
    py_text = re.sub(r"\bfalse\b", "False", py_text)
    py_text = re.sub(r"\bnull\b", "None", py_text)
    try:
        return ast.literal_eval(py_text)
    except (ValueError, SyntaxError):
        pass

    raise ValueError(
        "Could not parse input as JSON or JS-like object. "
        "Expected JSON, JSON Lines, `export default {...}`, or `const x = {...}`."
    )


def has_label(item, label_parts):
    label = clean_text((item or {}).get("_label")).lower()
    return any(part.lower() in label for part in label_parts)


def find_identifier_content(obj, label_parts):
    for entry in as_list(obj.get("identified_by")):
        if has_label(entry, label_parts):
            content = clean_text(entry.get("content"))
            if content:
                return content
    return ""


def find_name_content(obj, label_parts):
    for entry in as_list(obj.get("identified_by")):
        if entry.get("type") == "Name" and has_label(entry, label_parts):
            content = clean_text(entry.get("content"))
            if content:
                return content
    return ""


def collect_referred_content(obj, label_parts=None):
    values = []
    for entry in as_list(obj.get("referred_to_by")):
        if label_parts and not has_label(entry, label_parts):
            continue
        content = clean_text(entry.get("content"))
        if content:
            values.append(content)
    return unique_preserve(values)


def collect_classification_labels(obj):
    labels = []
    for item in as_list(obj.get("classified_as")):
        lbl = clean_text(item.get("_label"))
        if lbl:
            labels.append(lbl)
    return unique_preserve(labels)


def get_primary_title(obj):
    for entry in as_list(obj.get("identified_by")):
        if entry.get("type") != "Name":
            continue
        class_labels = [clean_text(c.get("_label")).lower() for c in as_list(entry.get("classified_as"))]
        if "preferred term" in class_labels or "primary title" in class_labels:
            content = clean_text(entry.get("content"))
            if content:
                return content

    for entry in as_list(obj.get("identified_by")):
        if entry.get("type") == "Name":
            content = clean_text(entry.get("content"))
            if content:
                return content

    return clean_text(obj.get("_label"))


def get_date_created(obj):
    produced_by = obj.get("produced_by") or {}
    timespan = produced_by.get("timespan") or {}

    for name in as_list(timespan.get("identified_by")):
        display = clean_text(name.get("content"))
        if display:
            return display

    begin = clean_text(timespan.get("begin_of_the_begin"))
    end = clean_text(timespan.get("end_of_the_end"))
    begin_year = extract_year(begin)
    end_year = extract_year(end)

    if begin_year and end_year and begin_year != end_year:
        return f"{begin_year}-{end_year}"
    return begin_year or end_year


def get_date_acquired_or_updated(obj):
    accession = find_identifier_content(obj, ["Accession Number"])
    year = extract_year(accession)
    if year:
        return year

    for activity in as_list(obj.get("changed_ownership_through")):
        timespan = activity.get("timespan") or {}
        end_year = extract_year(timespan.get("end_of_the_end"))
        if end_year:
            return end_year

    return ""


def get_image_url(obj):
    for visual in as_list(obj.get("representation")):
        image_id = clean_text(visual.get("id"))
        if image_id:
            return image_id

    for visual in as_list(obj.get("shows")):
        image_id = clean_text(visual.get("id"))
        if image_id:
            return image_id

    return ""


def get_medium(obj):
    materials = collect_referred_content(obj, ["Materials Description"])
    object_type = collect_referred_content(obj, ["Object Type"])
    class_labels = [lbl for lbl in collect_classification_labels(obj) if lbl and lbl.lower() != "artwork"]

    return " | ".join(unique_preserve(materials + object_type + class_labels))


def get_object_descriptions(obj):
    markdown = []
    html_desc = []
    unlabeled = []

    for entry in as_list(obj.get("referred_to_by")):
        if not has_label(entry, ["Object Description"]):
            continue
        content = clean_text(entry.get("content"))
        if not content:
            continue
        fmt = clean_text(entry.get("format")).lower()
        if "markdown" in fmt:
            markdown.append(content)
        elif "html" in fmt:
            html_desc.append(content)
        else:
            unlabeled.append(content)

    preferred = markdown or html_desc or unlabeled
    return unique_preserve(preferred)


def get_alt_text(obj):
    descriptions = get_object_descriptions(obj)
    return descriptions[0] if descriptions else ""


def get_description(obj):
    accession = find_identifier_content(obj, ["Accession Number"])
    tms_id = find_identifier_content(obj, ["TMS ID"])
    dor_uuid = find_identifier_content(obj, ["DOR UUID"])
    slug = find_identifier_content(obj, ["Slug Identifier"])

    core_parts = []
    core_parts.extend(get_object_descriptions(obj))
    core_parts.extend(collect_referred_content(obj, ["Culture Statement"]))
    core_parts.extend(collect_referred_content(obj, ["Place Created"]))
    core_parts.extend(collect_referred_content(obj, ["Source Credit Line", "Credit Line"]))
    core_parts.extend(collect_referred_content(obj, ["Copyright Statement"]))

    for mark in as_list(obj.get("carries")):
        if has_label(mark, ["Object Markings"]):
            content = clean_text(mark.get("content"))
            if content:
                core_parts.append(f"Markings: {content}")

    produced_by = obj.get("produced_by") or {}
    core_parts.extend(collect_referred_content(produced_by, ["Producer", "Artist/Maker"]))

    for activity in as_list(obj.get("changed_ownership_through")):
        core_parts.extend(collect_referred_content(activity, ["Provenance", "Acquisition"]))
        for parent in as_list(activity.get("part_of")):
            core_parts.extend(collect_referred_content(parent, ["Provenance"]))

    dimensions = []
    for entry in as_list(obj.get("referred_to_by")):
        if has_label(entry, ["Dimensions Statement"]):
            content = clean_text(entry.get("content"))
            if content:
                dimensions.append(content)
    dimensions = unique_preserve(dimensions)
    if dimensions:
        core_parts.append("Dimensions: " + " | ".join(dimensions))

    links = []
    for entry in as_list(obj.get("subject_of")):
        link_id = clean_text(entry.get("id"))
        has_format = clean_text(entry.get("has_format"))
        if link_id:
            links.append(link_id)
        if has_format:
            links.append(has_format)
    links = unique_preserve(links)
    if links:
        core_parts.append("Links: " + " | ".join(links))

    identifier_bits = []
    if accession:
        identifier_bits.append(f"Accession: {accession}")
    if tms_id:
        identifier_bits.append(f"TMS ID: {tms_id}")
    if dor_uuid:
        identifier_bits.append(f"DOR UUID: {dor_uuid}")
    if slug:
        identifier_bits.append(f"Slug: {slug}")
    if identifier_bits:
        core_parts.append("Identifiers: " + " | ".join(identifier_bits))

    return smart_join(unique_preserve(core_parts))

In [23]:
def convert_getty_json_to_artworks(json_path, artist_id, institution_id=1, output_csv=None):
    data = load_loose_json(json_path)

    if isinstance(data, dict) and "data" in data:
        objects = as_list(data.get("data"))
    elif isinstance(data, list):
        objects = data
    else:
        objects = [data]

    records = []

    for obj in objects:
        records.append({
            "id": "",
            "artist": artist_id,
            "title": get_primary_title(obj),
            "alt_text": get_alt_text(obj),
            "description": get_description(obj),
            "date_created": get_date_created(obj),
            "date_acquired_or_updated": get_date_acquired_or_updated(obj),
            "institution": institution_id,
            "medium": get_medium(obj),
            "image_url": get_image_url(obj),
        })

    converted = pd.DataFrame(records, columns=[
        "id",
        "artist",
        "title",
        "alt_text",
        "description",
        "date_created",
        "date_acquired_or_updated",
        "institution",
        "medium",
        "image_url",
    ])

    dedupe_cols = ["artist", "title", "date_created", "date_acquired_or_updated", "image_url"]

    def dedupe_frame(df):
        df = df.copy()
        for col in dedupe_cols:
            df[f"__k_{col}"] = df[col].fillna("").astype(str).str.strip().str.lower()
        key_cols = [f"__k_{col}" for col in dedupe_cols]
        df = df.drop_duplicates(subset=key_cols, keep="last")
        return df.drop(columns=key_cols)

    converted = dedupe_frame(converted)

    if output_csv:
        if os.path.exists(output_csv):
            existing = pd.read_csv(output_csv)
            merged = pd.concat([existing, converted], ignore_index=True)
            merged = dedupe_frame(merged)
            merged.to_csv(output_csv, index=False)
        else:
            converted.to_csv(output_csv, index=False)

    return converted

In [ ]:
artist_name = "Stephanie Syjuco"

ARTIST_ID = 22
INSTITUTION_ID = 6
INPUT_JSON = "./exports/getty.json"
OUTPUT_CSV = f"./formatted-exports/{artist_name}-1.csv"

converted = convert_getty_json_to_artworks(
    json_path=INPUT_JSON,
    artist_id=ARTIST_ID,
    institution_id=INSTITUTION_ID,
    output_csv=OUTPUT_CSV,
)

print(converted.to_csv(index=False))

id,artist,title,alt_text,description,date_created,date_acquired_or_updated,institution,medium,image_url
,22,Pileup (Herbaria),"A collage composed of diverse naturalist archival sources, including photographs of bones, foliage, and crystal formations.","A collage composed of diverse naturalist archival sources, including photographs of bones, foliage, and crystal formations. American. United States. The J. Paul Getty Museum, Los Angeles. © Stephanie Syjuco. Markings: (Verso) lower center, white label typed in black ink:  ""Catherine/Clark/Gallery/ San Francisco/ 248 Utah Street/ San Francisco, CA 94103/ 415.399.1439/www.cclarkgallery.com/ Stepanie Syjuco/ Pileup (Herbaria), 2021/ Hand-assembled pigmented inkjet prints on Hahnemuhle/ Baryta/  Edition of 3 + 2AP; 2/3/ 48 x 36 inches framed/ SSy00396C;
(Verso) lower center, white label typed in black ink: ""Mark Ryan/ Fine Art Services/ 510.788.6227/ info@markryanfineart.com/ MARKRYANFINEART.COM/ 3403 Piedmont Avenue/ Suite 501/ Oakland, C